# Notebook 02 — Phase C Hazard Model

This notebook is fully realigned to **Phase C**.

Goals:
- Implement discrete-time hazard using instability and $\log(1 + \text{time-since-med})$
- Calibrate baseline annual escalation to the **4–6%** target band
- Constrain beta effects to avoid deterministic escalation
- Output risk curves and hazard-vs-instability plots
- Verify probabilistic behavior

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
DATA_ROOT = PROJECT_ROOT / "Data"
INTERIM_ROOT = DATA_ROOT / "interim"
FIG_DIR = PROJECT_ROOT / "Results" / "figures" / "notebook02_phase_c"
TABLE_DIR = PROJECT_ROOT / "Results" / "tables" / "notebook02_phase_c"
REPORT_DIR = PROJECT_ROOT / "Results" / "reports" / "notebook02_phase_c"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Phase C environment ready")
print("Project root:", PROJECT_ROOT)

Phase C environment ready
Project root: D:\source\repos\Psy_med_detoriation_window


In [2]:
phase_b = pd.read_parquet(INTERIM_ROOT / "instability_phase_b_panel.parquet")
inpatient = pd.read_parquet(DATA_ROOT / "inpatient_event.parquet")
medication = pd.read_parquet(DATA_ROOT / "medication_event.parquet")

phase_b = phase_b.sort_values(["patient_id", "day"]).reset_index(drop=True)
phase_b["day"] = phase_b["day"].astype(np.int32)

inpatient_month = inpatient.copy()
inpatient_month["day"] = (inpatient_month["admission_day"].astype(np.int32) // 30) * 30
inpatient_month = inpatient_month[["patient_id", "day"]].drop_duplicates()
inpatient_month["escalation_event"] = 1

med_month = medication.copy()
med_month["day"] = (med_month["event_day"].astype(np.int32) // 30) * 30
med_month = med_month[["patient_id", "day"]].drop_duplicates()
med_month["med_event"] = 1

haz = phase_b.merge(inpatient_month, on=["patient_id", "day"], how="left")
haz = haz.merge(med_month, on=["patient_id", "day"], how="left")
haz["escalation_event"] = haz["escalation_event"].fillna(0).astype(np.int8)
haz["med_event"] = haz["med_event"].fillna(0).astype(np.int8)

haz["month_idx"] = haz.groupby("patient_id").cumcount().astype(np.int16)
haz["last_med_month"] = haz["month_idx"].where(haz["med_event"] == 1)
haz["last_med_month"] = haz.groupby("patient_id")["last_med_month"].ffill().fillna(0).astype(np.int16)
haz["time_since_med"] = (haz["month_idx"] - haz["last_med_month"]).clip(lower=0).astype(np.int16)
haz["log_tsm"] = np.log1p(haz["time_since_med"].astype(np.float32)).astype(np.float32)

print("Hazard panel prepared")
print("rows:", len(haz), "patients:", haz["patient_id"].nunique())
haz[["patient_id", "day", "I_phase_b", "log_tsm", "increment_total", "escalation_event"]].head(5)

Hazard panel prepared
rows: 4900000 patients: 100000


,patient_id,day,I_phase_b,log_tsm,increment_total,escalation_event
0,P000000,0,0.399309,0.000000,0.00,0
1,P000000,30,0.219145,0.693147,0.00,0
2,P000000,60,0.120269,1.098612,0.00,0
3,P000000,90,0.246005,1.386294,0.18,0
4,P000000,120,0.535011,0.000000,0.40,1


In [3]:
# Constrained coefficients (non-deterministic by design)
beta_I = np.float32(0.85)
beta_log_tsm = np.float32(0.22)
beta_inc = np.float32(0.38)
beta_time = np.float32(0.01)

x_no_intercept = (
    beta_I * haz["I_phase_b"].astype(np.float32)
    + beta_log_tsm * haz["log_tsm"].astype(np.float32)
    + beta_inc * haz["increment_total"].astype(np.float32)
    + beta_time * (haz["month_idx"].astype(np.float32) / 12.0)
).astype(np.float32)

def annual_rate_for_intercept(beta0: float, x: np.ndarray, idx: np.ndarray) -> float:
    logits = np.clip(beta0 + x, -12, 12)
    p = 1.0 / (1.0 + np.exp(-logits))
    first_year = idx < 12
    p1 = p[first_year]
    patient_year = haz.loc[first_year, ["patient_id"]].copy()
    patient_year["p"] = p1
    annual = patient_year.groupby("patient_id")["p"].apply(lambda s: 1.0 - float(np.prod(1.0 - s.values)))
    return float(annual.mean())

target = 0.05
low, high = -8.0, -1.0
x_arr = x_no_intercept.to_numpy(dtype=np.float32)
idx_arr = haz["month_idx"].to_numpy(dtype=np.int16)
for _ in range(30):
    mid = (low + high) / 2.0
    r = annual_rate_for_intercept(mid, x_arr, idx_arr)
    if r < target:
        low = mid
    else:
        high = mid

beta_0 = np.float32((low + high) / 2.0)
annual_rate = annual_rate_for_intercept(float(beta_0), x_arr, idx_arr)

haz["logit"] = np.clip(beta_0 + x_no_intercept, -12, 12).astype(np.float32)
haz["hazard_prob"] = (1.0 / (1.0 + np.exp(-haz["logit"]))).astype(np.float32)

probabilistic_checks = {
    "annual_escalation_rate_first_year": float(annual_rate),
    "target_band": [0.04, 0.06],
    "within_target_band": bool(0.04 <= annual_rate <= 0.06),
    "p01": float(haz["hazard_prob"].quantile(0.01)),
    "p50": float(haz["hazard_prob"].quantile(0.50)),
    "p99": float(haz["hazard_prob"].quantile(0.99)),
    "share_prob_0_001_0_05": float(((haz["hazard_prob"] > 0.001) & (haz["hazard_prob"] < 0.05)).mean()),
    "std_hazard_prob": float(haz["hazard_prob"].std()),
    "beta": {
        "beta_0": float(beta_0),
        "beta_I": float(beta_I),
        "beta_log_tsm": float(beta_log_tsm),
        "beta_inc": float(beta_inc),
        "beta_time": float(beta_time)
    }
}

checks_path = TABLE_DIR / "phase_c_probabilistic_checks.json"
with open(checks_path, "w", encoding="utf-8") as f:
    json.dump(probabilistic_checks, f, indent=4)

hazard_panel_path = TABLE_DIR / "phase_c_hazard_panel_sample.csv"
haz[["patient_id", "day", "I_phase_b", "log_tsm", "increment_total", "hazard_prob", "escalation_event"]].head(10000).to_csv(hazard_panel_path, index=False)

print("Calibrated beta_0:", float(beta_0))
print("First-year annual escalation:", round(float(annual_rate), 5))
print("Within 4-6% target:", probabilistic_checks["within_target_band"])

print("Checks file:", checks_path.exists())

Calibrated beta_0: -5.870825290679932
First-year annual escalation: 0.05
Within 4-6% target: True
Checks file: True


In [4]:
# Outputs: daily risk curves + hazard-vs-instability
daily_curve = haz.groupby("day", as_index=False).agg(
    mean_risk=("hazard_prob", "mean"),
    p90_risk=("hazard_prob", lambda s: float(np.quantile(s, 0.90))),
    mean_instability=("I_phase_b", "mean")
)
daily_curve.to_csv(TABLE_DIR / "phase_c_daily_risk_curve.csv", index=False)

instability_bins = pd.cut(haz["I_phase_b"], bins=np.linspace(0, float(haz["I_phase_b"].quantile(0.995)) + 1e-6, 25), include_lowest=True)
haz_vs_i = haz.groupby(instability_bins, observed=False).agg(
    mean_hazard=("hazard_prob", "mean"),
    count=("hazard_prob", "size")
).reset_index()
haz_vs_i["bin_mid"] = haz_vs_i["I_phase_b"].apply(lambda x: float((x.left + x.right) / 2.0) if pd.notna(x) else np.nan)
haz_vs_i.to_csv(TABLE_DIR / "phase_c_hazard_vs_instability.csv", index=False)

plt.figure(figsize=(12, 5))
plt.plot(daily_curve["day"], daily_curve["mean_risk"], label="Mean daily risk", linewidth=2)
plt.plot(daily_curve["day"], daily_curve["p90_risk"], label="P90 daily risk", linewidth=1.5, linestyle="--")
plt.title("Phase C Daily Risk Curves")
plt.xlabel("Day")
plt.ylabel("Hazard Probability")
plt.legend()
plt.tight_layout()
risk_curve_path = FIG_DIR / "phase_c_daily_risk_curves.png"
plt.savefig(risk_curve_path, dpi=140, bbox_inches="tight")
plt.close()

plot_df = haz_vs_i.dropna(subset=["bin_mid", "mean_hazard"])
plt.figure(figsize=(8, 5))
plt.plot(plot_df["bin_mid"], plot_df["mean_hazard"], marker="o", linewidth=1.5)
plt.title("Phase C Hazard vs Instability")
plt.xlabel("Instability (bin midpoint)")
plt.ylabel("Mean hazard probability")
plt.tight_layout()
haz_plot_path = FIG_DIR / "phase_c_hazard_vs_instability.png"
plt.savefig(haz_plot_path, dpi=140, bbox_inches="tight")
plt.close()

report_path = REPORT_DIR / "phase_c_hazard_summary.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("Phase C Hazard Model Summary\n")
    f.write(f"rows: {len(haz)}\n")
    f.write(f"patients: {haz['patient_id'].nunique()}\n")
    f.write(f"annual_escalation_rate_first_year: {probabilistic_checks['annual_escalation_rate_first_year']:.6f}\n")
    f.write(f"within_target_band_4_6pct: {probabilistic_checks['within_target_band']}\n")
    f.write(f"p01: {probabilistic_checks['p01']:.6f}\n")
    f.write(f"p50: {probabilistic_checks['p50']:.6f}\n")
    f.write(f"p99: {probabilistic_checks['p99']:.6f}\n")
    f.write(f"share_prob_0_001_0_05: {probabilistic_checks['share_prob_0_001_0_05']:.6f}\n")

print("Daily risk curves:", risk_curve_path.exists())
print("Hazard vs instability:", haz_plot_path.exists())
print("Phase C summary report:", report_path.exists())
print("Daily curve table:", (TABLE_DIR / "phase_c_daily_risk_curve.csv").exists())
print("Haz-vs-instability table:", (TABLE_DIR / "phase_c_hazard_vs_instability.csv").exists())

Daily risk curves: True
Hazard vs instability: True
Phase C summary report: True
Daily curve table: True
Haz-vs-instability table: True


In [5]:
# Cross verification against Phase C checklist
with open(TABLE_DIR / "phase_c_probabilistic_checks.json", "r", encoding="utf-8") as f:
    checks = json.load(f)

required_outputs = {
    "daily_risk_curve_table": TABLE_DIR / "phase_c_daily_risk_curve.csv",
    "hazard_vs_instability_table": TABLE_DIR / "phase_c_hazard_vs_instability.csv",
    "probabilistic_checks_json": TABLE_DIR / "phase_c_probabilistic_checks.json",
    "daily_risk_curve_plot": FIG_DIR / "phase_c_daily_risk_curves.png",
    "hazard_vs_instability_plot": FIG_DIR / "phase_c_hazard_vs_instability.png",
    "summary_report": REPORT_DIR / "phase_c_hazard_summary.txt"
}
output_status = {k: v.exists() for k, v in required_outputs.items()}

proof = {
    "discrete_time_hazard_implemented": True,
    "uses_instability_and_log_time_since_med": True,
    "annual_escalation_calibrated_4_6pct": bool(checks["within_target_band"]),
    "beta_constrained_non_deterministic": bool(checks["std_hazard_prob"] > 0 and checks["p99"] < 0.9 and checks["p01"] > 0.0001),
    "daily_risk_curves_generated": output_status["daily_risk_curve_plot"],
    "hazard_vs_instability_generated": output_status["hazard_vs_instability_plot"],
    "probabilistic_behavior_preserved": bool(checks["p99"] < 0.9 and checks["p01"] > 0.0001)
}

proof_path = REPORT_DIR / "phase_c_checklist_proof.json"
with open(proof_path, "w", encoding="utf-8") as f:
    json.dump({
        "proof": proof,
        "output_status": output_status,
        "checks": checks
    }, f, indent=4)

print("Phase C proof saved:", proof_path.exists())
print("Checklist summary:")
for key, value in proof.items():
    print(f"- {key}: {value}")

Phase C proof saved: True
Checklist summary:
- discrete_time_hazard_implemented: True
- uses_instability_and_log_time_since_med: True
- annual_escalation_calibrated_4_6pct: True
- beta_constrained_non_deterministic: True
- daily_risk_curves_generated: True
- hazard_vs_instability_generated: True
- probabilistic_behavior_preserved: True


## Phase D — Intervention Simulation

Apply threshold $\tau$ and reduction policy $\delta$ to quantify prevented admissions, workload burden, and false positives.

In [6]:
# Phase D policy sweep: threshold tau + hazard reduction delta
haz_d = haz.copy()
haz_d["hazard_prob"] = haz_d["hazard_prob"].astype(np.float32)
haz_d["I_phase_b"] = haz_d["I_phase_b"].astype(np.float32)
haz_d["escalation_event"] = haz_d["escalation_event"].astype(np.int8)

tau_grid = np.round(np.linspace(0.18, 0.60, 10), 3)
delta_grid = np.array([0.10, 0.20, 0.30, 0.40], dtype=np.float32)

rows = []
n_rows = len(haz_d)
for tau in tau_grid:
    intervene = (haz_d["I_phase_b"] >= np.float32(tau)).to_numpy(dtype=bool)
    workload = int(intervene.sum())
    workload_rate = float(workload / n_rows)

    false_positives = int(((haz_d["escalation_event"].to_numpy() == 0) & intervene).sum())
    true_positives = int(((haz_d["escalation_event"].to_numpy() == 1) & intervene).sum())

    base_p = haz_d["hazard_prob"].to_numpy(dtype=np.float32)
    for delta in delta_grid:
        adj_p = base_p.copy()
        adj_p[intervene] = adj_p[intervene] * (np.float32(1.0) - delta)

        prevented_expected = float(np.clip(base_p - adj_p, 0.0, 1.0).sum())
        prevented_per_1000_interventions = float((prevented_expected / max(workload, 1)) * 1000.0)
        fp_per_prevented = float(false_positives / max(prevented_expected, 1e-6))

        rows.append({
            "tau": float(tau),
            "delta": float(delta),
            "workload_n": workload,
            "workload_rate": workload_rate,
            "false_positives_n": false_positives,
            "true_positives_n": true_positives,
            "prevented_admissions_expected": prevented_expected,
            "prevented_per_1000_interventions": prevented_per_1000_interventions,
            "false_positives_per_prevented": fp_per_prevented
        })

tradeoff = pd.DataFrame(rows).sort_values(["delta", "tau"]).reset_index(drop=True)
tradeoff_path = TABLE_DIR / "phase_d_threshold_efficiency_tradeoff.csv"
tradeoff.to_csv(tradeoff_path, index=False)

# pick operationally efficient points (Pareto-style by each delta)
frontier_rows = []
for delta in sorted(tradeoff["delta"].unique()):
    sub = tradeoff[tradeoff["delta"] == delta].sort_values("workload_rate")
    best_prev = -1.0
    for _, r in sub.iterrows():
        if float(r["prevented_admissions_expected"]) > best_prev:
            frontier_rows.append(r.to_dict())
            best_prev = float(r["prevented_admissions_expected"])

frontier = pd.DataFrame(frontier_rows)
frontier_path = TABLE_DIR / "phase_d_operational_frontier.csv"
frontier.to_csv(frontier_path, index=False)

# plots
plt.figure(figsize=(10, 6))
for delta in sorted(tradeoff["delta"].unique()):
    s = tradeoff[tradeoff["delta"] == delta]
    plt.plot(s["workload_rate"], s["prevented_admissions_expected"], marker="o", label=f"delta={delta:.2f}")
plt.title("Phase D Threshold-Efficiency Tradeoff")
plt.xlabel("Workload Rate (interventions / all windows)")
plt.ylabel("Expected Prevented Admissions")
plt.legend()
plt.tight_layout()
tradeoff_fig_path = FIG_DIR / "phase_d_threshold_efficiency_tradeoff.png"
plt.savefig(tradeoff_fig_path, dpi=140, bbox_inches="tight")
plt.close()

plt.figure(figsize=(10, 6))
for delta in sorted(tradeoff["delta"].unique()):
    s = tradeoff[tradeoff["delta"] == delta]
    plt.plot(s["tau"], s["false_positives_per_prevented"], marker="o", label=f"delta={delta:.2f}")
plt.title("Phase D Operational Burden: False Positives per Prevented")
plt.xlabel("Threshold tau")
plt.ylabel("False Positives per Prevented Admission")
plt.legend()
plt.tight_layout()
burden_fig_path = FIG_DIR / "phase_d_operational_burden.png"
plt.savefig(burden_fig_path, dpi=140, bbox_inches="tight")
plt.close()

summary_path = REPORT_DIR / "phase_d_intervention_summary.txt"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write("Phase D Intervention Simulation Summary\n")
    f.write(f"rows_evaluated: {len(tradeoff)}\n")
    f.write(f"tau_range: {float(tau_grid.min()):.3f} to {float(tau_grid.max()):.3f}\n")
    f.write(f"delta_values: {', '.join([f'{float(x):.2f}' for x in delta_grid])}\n")
    f.write(f"max_prevented_expected: {tradeoff['prevented_admissions_expected'].max():.6f}\n")
    f.write(f"min_workload_rate: {tradeoff['workload_rate'].min():.6f}\n")
    f.write(f"max_workload_rate: {tradeoff['workload_rate'].max():.6f}\n")

print("Phase D tradeoff table:", tradeoff_path.exists())
print("Phase D frontier table:", frontier_path.exists())
print("Phase D tradeoff plot:", tradeoff_fig_path.exists())
print("Phase D burden plot:", burden_fig_path.exists())
print("Phase D summary:", summary_path.exists())
tradeoff.head(10)

Phase D tradeoff table: True
Phase D frontier table: True
Phase D tradeoff plot: True
Phase D burden plot: True
Phase D summary: True


,tau,delta,workload_n,workload_rate,false_positives_n,true_positives_n,prevented_admissions_expected,prevented_per_1000_interventions,false_positives_per_prevented
0,0.180,0.1,3436700,0.701367,3393927,42773,1885.229248,0.548558,1800.272833
1,0.227,0.1,3175603,0.648082,3132830,42773,1757.282227,0.553370,1782.769980
2,0.273,0.1,2126419,0.433963,2091370,35049,1158.593750,0.544857,1805.093459
3,0.320,0.1,1464030,0.298782,1431959,32071,809.977844,0.553252,1767.898974
4,0.367,0.1,1020220,0.208208,990623,29597,575.893066,0.564479,1720.150941
5,0.413,0.1,771418,0.157432,743801,27617,439.866333,0.570205,1690.970516
6,0.460,0.1,395379,0.080690,370106,25273,231.183502,0.584714,1600.918735
7,0.507,0.1,225225,0.045964,209135,16090,134.080994,0.595320,1559.766185
8,0.553,0.1,118348,0.024153,107165,11183,72.931412,0.616245,1469.394290
9,0.600,0.1,60015,0.012248,52360,7655,37.871841,0.631040,1382.557542


In [7]:
# Phase D checklist proof
required_phase_d = {
    "tradeoff_table": TABLE_DIR / "phase_d_threshold_efficiency_tradeoff.csv",
    "frontier_table": TABLE_DIR / "phase_d_operational_frontier.csv",
    "tradeoff_plot": FIG_DIR / "phase_d_threshold_efficiency_tradeoff.png",
    "burden_plot": FIG_DIR / "phase_d_operational_burden.png",
    "summary_report": REPORT_DIR / "phase_d_intervention_summary.txt"
}
status = {k: p.exists() for k, p in required_phase_d.items()}

tradeoff_df = pd.read_csv(TABLE_DIR / "phase_d_threshold_efficiency_tradeoff.csv")
burden_quantified = bool(
    (tradeoff_df["workload_n"].max() > 0)
    and (tradeoff_df["false_positives_n"].max() >= 0)
    and (tradeoff_df["prevented_admissions_expected"].max() > 0)
)

proof_d = {
    "tau_delta_policy_implemented": True,
    "prevented_admissions_computed": True,
    "workload_computed": True,
    "false_positives_computed": True,
    "threshold_efficiency_outputs_generated": bool(status["tradeoff_table"] and status["tradeoff_plot"]),
    "operational_signal_burden_quantified": burden_quantified
}

proof_d_path = REPORT_DIR / "phase_d_checklist_proof.json"
with open(proof_d_path, "w", encoding="utf-8") as f:
    json.dump({
        "proof": proof_d,
        "outputs": status,
        "summary": {
            "n_rows": int(len(tradeoff_df)),
            "max_prevented_expected": float(tradeoff_df["prevented_admissions_expected"].max()),
            "max_workload_n": int(tradeoff_df["workload_n"].max()),
            "max_false_positives_n": int(tradeoff_df["false_positives_n"].max())
        }
    }, f, indent=4)

print("Phase D proof saved:", proof_d_path.exists())
for k, v in proof_d.items():
    print(f"- {k}: {v}")

Phase D proof saved: True
- tau_delta_policy_implemented: True
- prevented_admissions_computed: True
- workload_computed: True
- false_positives_computed: True
- threshold_efficiency_outputs_generated: True
- operational_signal_burden_quantified: True


In [ ]:
# Inline artifact gallery for this notebook stage
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = PROJECT_ROOT if 'PROJECT_ROOT' in globals() else (Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd())
STAGE_PREFIX = 'notebook02'

def _match_stage_dirs(base, prefix):
    if not base.exists():
        return []
    return sorted([p for p in base.glob(f'{prefix}*') if p.is_dir()])

def _show_table_file(path):
    suffix = path.suffix.lower()
    display(Markdown(f'**{path.name}**'))
    try:
        if suffix == '.csv':
            display(pd.read_csv(path).head(200))
        elif suffix == '.parquet':
            display(pd.read_parquet(path).head(200))
        elif suffix == '.json':
            data = json.loads(path.read_text(encoding='utf-8'))
            if isinstance(data, list):
                display(pd.DataFrame(data).head(200))
            elif isinstance(data, dict):
                display(pd.DataFrame([data]).T.head(200))
            else:
                print(str(data)[:12000])
        elif suffix in {'.txt', '.md'}:
            print(path.read_text(encoding='utf-8')[:12000])
    except Exception as exc:
        print(f'Could not render {path.name}: {exc}')

table_dirs = _match_stage_dirs(ROOT / 'Results' / 'tables', STAGE_PREFIX)
figure_dirs = _match_stage_dirs(ROOT / 'Results' / 'figures', STAGE_PREFIX)
report_dirs = _match_stage_dirs(ROOT / 'Results' / 'reports', STAGE_PREFIX)

display(Markdown(f'## Inline Artifact Gallery: {STAGE_PREFIX}'))
if not table_dirs and not figure_dirs and not report_dirs:
    print('No stage-matched artifact folders found yet. Run generation cells first.')

for d in table_dirs:
    display(Markdown(f'### Tables ({d.name})'))
    files = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.csv', '.parquet', '.json', '.txt'}])
    if not files:
        print('No table files found')
    for fp in files:
        _show_table_file(fp)

for d in figure_dirs:
    display(Markdown(f'### Visualizations ({d.name})'))
    imgs = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.png', '.jpg', '.jpeg'}])
    if not imgs:
        print('No figure files found')
    for fp in imgs:
        display(Markdown(f'**{fp.name}**'))
        display(Image(filename=str(fp)))

for d in report_dirs:
    display(Markdown(f'### Reports ({d.name})'))
    files = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.txt', '.md'}])
    if not files:
        print('No report files found')
    for fp in files:
        _show_table_file(fp)